# Statistical Modeling and Inferencing — Assignment 1

**Chosen Dataset:** Dataset 1 — `Diamonds Dataset` (~53,940 observations; predicting price)

**Assignment focus:** Data Exploration and Preparation → Regression Modelling and Validation → Interpretation and Insights.

## Summary

This report analyses the **Diamonds dataset** (53,940 observations, 10 variables) to understand which diamond
characteristics drive price and to build an explanatory regression model. After cleaning (removing **146** duplicate rows
and treating physically impossible zero values in x, y, z as missing), the data were explored through descriptive
statistics, distribution plots, boxplots by category, one-way ANOVA, and Pearson correlation analysis. `Carat`
emerged as by far the **strongest predictor** of price (Pearson r ≈ 0.92), and cut, color, and clarity all showed statistically
significant differences in mean price across their categories — although, perhaps counter-intuitively, the best-graded
categories do not have the highest median price in the raw data, a pattern explained later in this report.

A Variance Inflation Factor (VIF) analysis revealed **severe multicollinearity** among carat, x, y, and z, since all four
measure aspects of diamond size. Three regression models were built and compared on a held-out 20% test set: a full
model with all predictors, a reduced model that removes the redundant dimension variables (x, y, z), and a log-price
model to address the right-skew in price. The full model gave the best test-set performance (R² = 0.918, RMSE ≈
₹1,141), narrowly ahead of the reduced model (R² = 0.914); the log-price model fit the log scale well (R² = 0.889) but
performed far worse than simply predicting the mean once its predictions were converted back to the original price
scale (test R² = -3.146) — a striking illustration of retransformation bias.


The `Model 1` (Full Model) was selected as the **final** model. carat is the single most influential predictor (≈ ₹11,640 in price per
additional carat, holding other variables fixed), followed by clarity and color, while depth and table have comparatively
small effects. Residual diagnostics show heteroscedasticity, and **Cook’s-distance** identifies a small number of highly
influential, high-priced observations that merit further business investigation rather than automatic removal.

## Part 1: Data Exploration and Preparation

In [0]:
%pip install statsmodels scikit-learn

In [0]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Set Defaults
sns.set_style("whitegrid")
np.random.seed(46)

### Load the dataset

Load the diamond dataset from Github and check it's shape and attributes

In [0]:
# Load the diamonds dataset from GitHub
DATA_PATH = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/diamonds.csv"
df = pd.read_csv(DATA_PATH)

# Display basic information
print("Dataset loaded successfully")
print("Shape:", df.shape)
df.head()

### Basic structure and descriptive statistics

We first inspect the number of observations, data types, unique values and descriptive statistics. This helps identify numerical and categorical variables before modelling.

In [0]:
# Display dataset dimensions
print("Shape:", df.shape)

# Check data types and missing values
print("\nData types and non-null counts:")
df.info()

# Calculate descriptive statistics for numerical variables
print("\nDescriptive statistics:")
print(df.describe().T.to_string())

# Count unique values per column
print("\nNumber of unique values:")
print(df.nunique())

### Insight
The dataset has **53,940 observations and 10 variables**. There are 7 numerical variables and 3 categorical variables. `price` is the target variable. The descriptive statistics also show that `price`, `carat`, and the dimension variables have noticeable spread/skewness, so their distributions need further investigation before modelling.


### Data quality checks

We check for missing values, duplicate rows and impossible physical measurements. In the diamond data, `x`, `y` and `z` represent dimensions, so a value of zero is not physically meaningful and should be treated as missing.

In [0]:
# Check for missing values
print("Missing values before cleaning:")
print(df.isnull().sum())
print()

# Check for duplicate rows
print("Number of duplicate rows:", df.duplicated().sum())
print()

# Check for zero values in physical dimensions (invalid)
print("Zero values in dimensions:")
print((df[["x", "y", "z"]] == 0).sum())

### Insight
There are **no missing values**, but **146 duplicate rows** are present. Zero values occur in `x`, `y`, and `z`; these are invalid because they represent physical diamond dimensions. Therefore, duplicates should be removed and zero dimensions should be treated as missing and imputed.


### Handle invalid values

Zero values in `x`, `y` and `z` are treated as missing because a diamond cannot have zero physical dimensions. For the regression analysis, these missing numerical values are replaced with the median of the respective variable. Duplicate observations are removed. We do not want to automatically delete or clip price outliers because very expensive diamonds may be genuine observations.

In [0]:
# Remove duplicate rows
df = df.drop_duplicates().copy()

# Replace zero values with NaN, then impute with median
for col in ["x", "y", "z"]:
    df.loc[df[col] == 0, col] = np.nan
    df[col] = df[col].fillna(df[col].median())

# Verify no missing values remain
print("Missing values after cleaning:")
print(df.isnull().sum())
print("Shape after removing duplicates:", df.shape)

### Insight
After cleaning, there are **53,794 observations** and no missing values. The zero values in `x`, `y`, and `z` were replaced with their respective medians. This keeps the observations while avoiding impossible physical measurements.


### Categorical variables

`cut`, `color` and `clarity` are categorical predictors. They will be converted into dummy variables automatically in the regression models using `C(variable)` in the statistical formula. This avoids assigning artificial numerical meaning to the categories.

In [0]:
# Define categorical and numerical columns
categorical_cols = ["cut", "color", "clarity"]
numerical_cols = ["carat", "depth", "table", "x", "y", "z", "price"]

# Display frequency counts for each categorical variable
for col in categorical_cols:
    print(f"\n{col}:")
    print(df[col].value_counts().to_frame("Count"))

### Insight
`cut`, `color`, and `clarity` are categorical variables with multiple meaningful levels. They should be **retained as predictors** because these characteristics describe diamond quality and should not be treated as continuous numbers. Dummy/treatment coding is therefore appropriate for regression.


### Distribution of numerical variables
Histograms are used to understand the shape and skewness of the numerical variables.

In [0]:
# Plot histograms for all numerical variables
df[numerical_cols].hist(figsize=(16, 10), bins=30)
plt.suptitle("Distributions of Numerical Variables", fontsize=16)
plt.tight_layout()
plt.show()

# Calculate and display skewness for each variable
print("Skewness by variable:")
skewness_values = df[numerical_cols].skew().sort_values(ascending=False)
for var, skew_val in skewness_values.items():
    print(f"  {var}: {skew_val:.4f}")

### Insight
`price` is positively skewed (skewness ≈ **1.62**), while `y` and `z` are also strongly skewed. `carat` is positively skewed as well. Therefore, the price distribution should be considered when interpreting regression results, and a log-price model is reasonable to test.


### Categorical distributions
Bar charts show how observations are distributed across `cut`, `color` and `clarity`.

In [0]:
# Create bar charts for categorical variable distributions
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, col in zip(axes, categorical_cols):
    sns.countplot(data=df, x=col, ax=ax)
    ax.set_title(f"Distribution of {col}")
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

### Insight
The categorical variables are not equally distributed across their categories; for example, `Ideal` is the most common `cut`, while some categories occur less frequently. The categories are still retained because they represent meaningful diamond characteristics, but their unequal frequencies should be kept in mind when interpreting group comparisons.


### Price across categorical groups
These plots help identify whether diamond price differs across cut, color and clarity categories.

In [0]:
# Create boxplots to compare price across categorical groups
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, col in zip(axes, categorical_cols):
    sns.boxplot(data=df, x=col, y="price", ax=ax)
    ax.set_title(f"Price by {col}")
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

### Insight
The boxplots show that diamond prices differ across `cut`, `color`, and `clarity`. This suggests that these categorical variables may contain useful information for predicting price and should be tested statistically before modelling.


### Statistical comparison of price across categories

A one-way ANOVA is used to test whether mean price differs across the groups of each categorical variable. A small p-value indicates evidence that at least one group mean differs. Because the dataset is large, statistical significance should be interpreted together with the visual differences.

In [0]:
# Perform one-way ANOVA to test if mean price differs across groups
for col in categorical_cols:
    groups = [g["price"].values for _, g in df.groupby(col)]
    f_stat, p_value = stats.f_oneway(*groups)
    print(f"{col}: F-statistic = {f_stat:.3f}, p-value = {p_value:.4g}")

### Insight
The ANOVA results show **statistically significant differences in mean price across all three categorical variables** (`cut`, `color`, and `clarity`; p < 0.001). Therefore, all three categorical variables are relevant to the modelling stage and should be retained.


### Outlier identification

Boxplots and the IQR rule are used to identify potential outliers. These observations are documented rather than automatically removed because some high-priced diamonds can be genuine and important for the business question.

In [0]:
# Identify outliers using IQR method (1.5 * IQR rule)
outlier_summary = []
for col in numerical_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_summary.append([col, lower, upper, count])

# Display outlier statistics
outlier_summary = pd.DataFrame(outlier_summary, columns=["Variable", "Lower Bound", "Upper Bound", "Outlier Count"])
print(outlier_summary.to_string())

### Insight
Outliers are present, especially in `price` (**3,523 observations**) and `carat` (**1,873
observations**). These are not automatically removed because some may represent genuinely
expensive or large diamonds. The outlier analysis therefore supports investigating their
influence rather than deleting them mechanically.



### Correlation analysis

Pearson correlation measures linear association. We focus on relationships with the target variable, `price`.

In [0]:
# Calculate Pearson correlation matrix for numerical variables
corr_cols = ["price", "carat", "depth", "table", "x", "y", "z"]
pearson_corr = df[corr_cols].corr(method="pearson")

# Visualize correlations with heatmap
plt.figure(figsize=(10, 7))
sns.heatmap(pearson_corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Pearson Correlation Matrix")
plt.show()

### Insight
`carat` has the strongest Pearson correlation with `price` (**0.922**). `x`, `y`, and `z` also have very strong positive correlations with price (about **0.87–0.89**), while `table` has a weak positive correlation (**0.127**) and `depth` has almost no simple linear correlation (**−0.011**).

However, **high correlation with price alone does not mean all variables should be used together**. `x`, `y`, and `z` are measurements of diamond dimensions and are highly related to each other, so they may create multicollinearity. `carat` is the most important numerical predictor, while `depth` and `table` should be evaluated through the regression model rather than removed solely from their simple correlations.


### Relationships between important numerical predictors and price

In [0]:
# Create scatterplots to visualize relationships with price
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, col in zip(axes.ravel(), ["carat", "depth", "table", "x", "y", "z"]):
    sns.scatterplot(data=df, x=col, y="price", alpha=0.25, s=20, ax=ax)
    ax.set_title(f"Price vs {col}")
plt.tight_layout()
plt.show()

### Insight
The scatterplots confirm a strong positive relationship between `carat` and `price`, and similar relationships between the physical dimensions (`x`, `y`, `z`) and price. The relationships for `depth` and `table` are much weaker. This supports giving greater importance to `carat` and checking multicollinearity before deciding whether the dimension variables should enter the same model.


## Part 2: Model Development and Validation

The assignment asks for regression modelling, different predictor combinations or transformations, comparison using statistical metrics, multicollinearity checks and selection of a best model. We therefore will use three simple regression models:

1. **Model 1:** all numerical and categorical predictors.
2. **Model 2:** remove highly correlated physical-dimension variables (`x`, `y`, `z`) and retain the main predictors.
3. **Model 3:** use the same predictors as Model 2 but model `log(price)` to address the strong right-skewness in price.


### Multicollinearity check

In [0]:
# Calculate Variance Inflation Factor (VIF) to detect multicollinearity
vif_data = df[["carat", "depth", "table", "x", "y", "z"]].copy()
vif_results = pd.DataFrame({
    "Variable": vif_data.columns,
    "VIF": [variance_inflation_factor(vif_data.values, i) for i in range(vif_data.shape[1])]
})
print(vif_results.sort_values("VIF", ascending=False))

### Insight


The VIF results indicate **severe multicollinearity**. In particular, `x`, `y`, and `z` have extremely high VIF values, while `carat` also has a high VIF (**78.35**). This means these variables contain overlapping information about diamond size.


### Train-test split

An 80/20 split is used so that the regression models can be compared on observations that were not used to estimate their coefficients.

In [0]:
# Split data into 80% training and 20% testing sets
train_data, test_data = train_test_split(
    df, test_size=0.20, random_state=46
)
print("Training observations:", len(train_data))
print("Testing observations:", len(test_data))

### Insight
The data are split into **43,035 training observations (80%)** and **10,759 testing observations (20%)**. The test set is kept separate so that model performance can be evaluated on unseen observations.


### Model 1 — Full multiple linear regression

This model investigates the relationship between price and all available predictors. Categorical variables are encoded through treatment coding in the formula.

In [0]:
# Model 1: Full model with all predictors
formula1 = "price ~ carat + depth + table + x + y + z + C(cut) + C(color) + C(clarity)"
model1 = ols(formula1, data=train_data).fit()
print(model1.summary())

### Insight
Model 1 explains about **92.1% of the variation in price** on the training data. Most predictors and categorical levels are statistically significant (p < 0.05). However, `y` is **not statistically significant (p = 0.422)** after controlling for the other predictors, which is consistent with the strong multicollinearity among the dimension variables. The large condition number also indicates multicollinearity. Therefore, individual coefficients for `x`, `y`, and `z` should be interpreted cautiously.


### 16. Model 2 — Reduced multiple linear regression

Based on the correlation and multicollinearity analysis, the strongly related physical-dimension variables `x`, `y` and `z` are removed. The model retains `carat`, `depth`, `table`, `cut`, `color` and `clarity`.

In [0]:
# Model 2: Reduced model without x, y, z to avoid multicollinearity
formula2 = "price ~ carat + depth + table + C(cut) + C(color) + C(clarity)"
model2 = ols(formula2, data=train_data).fit()
print(model2.summary())

### Insight
After removing `x`, `y`, and `z`, Model 2 still explains about **91.6% of price variation**, only slightly below Model 1. All retained predictors (`carat`, `depth`, `table`) and the categorical variables are statistically significant (p < 0.05). This makes Model 2 **more interpretable and less affected by the direct dimensional variables**.


### 17. Model 3 — Log-price regression

Price is strongly right-skewed, so a log transformation can improve the linear-model fit and reduce the effect of extreme prices. The coefficients are interpreted as relationships with log price rather than raw price.

In [0]:
# Create log-transformed price variable
train_data = train_data.copy()
test_data = test_data.copy()
train_data["log_price"] = np.log1p(train_data["price"])
test_data["log_price"] = np.log1p(test_data["price"])

# Model 3: Log-price model to address skewness
formula3 = "log_price ~ carat + depth + table + C(cut) + C(color) + C(clarity)"
model3 = ols(formula3, data=train_data).fit()
print(model3.summary())

### Insight
The log-price model has an R² of **0.889**. In this specification, `depth` is not statistically significant (p = 0.963), while `carat`, `table`, `cut`, `color`, and `clarity` remain significant. Although the log transformation is statistically reasonable because price is skewed, its predictive performance must be compared with the other models before selecting it.


### 18. Compare the three models

The models are compared using **R², adjusted R², RMSE and MAE**. For the log-price model, predictions are converted back to the original price scale before calculating RMSE and MAE. Higher R²/adjusted R² and lower RMSE/MAE indicate better predictive performance.

In [0]:
# Define function to evaluate model performance on test set
def evaluate_model(model, formula, test_df, target="price", log_target=False):
    pred = model.predict(test_df)
    if log_target:
        pred = np.expm1(pred)  # Back-transform log predictions
    actual = test_df[target]
    return {
        "R2 (test)": r2_score(actual, pred),
        "RMSE (test)": np.sqrt(mean_squared_error(actual, pred)),
        "MAE (test)": mean_absolute_error(actual, pred),
        "Adjusted R2 (train)": model.rsquared_adj,
        "AIC (train)": model.aic,
        "BIC (train)": model.bic
    }

# Compare all three models
results = pd.DataFrame({
    "Model 1 - Full": evaluate_model(model1, formula1, test_data),
    "Model 2 - Reduced": evaluate_model(model2, formula2, test_data),
    "Model 3 - Log Price": evaluate_model(model3, formula3, test_data, log_target=True)
}).T

print(results.round(3).to_string())

### Insight
Model 1 gives the best test performance: **R² = 0.918, RMSE = 1140.73 and MAE = 739.64**. Model 2 is
slightly weaker (R² = 0.914, RMSE = 1164.19, MAE = 808.59). Model 3 (log-price), however, is not
merely "substantially worse" — once its predictions are converted back to the original price
scale with `expm1`, its **test R² is -3.146** (RMSE ≈ 8,104, MAE ≈ 1,670), i.e. it performs far
worse than simply predicting the mean price for every diamond. `AIC`/`BIC` for Model 3 are also **not directly comparable** to Models 1-2 in the
table above, since Model 3's dependent variable is `log_price` rather than `price` — AIC/BIC can
only be compared across models fit to the *same* response variable. Therefore, **Model 1 is
selected as the final predictive model**, while Model 2 remains useful as a simpler,
more interpretable alternative.


### Residual diagnostics for the selected model

The final model should have reasonably random residuals without a strong pattern. We also inspect the Q-Q
plot to assess whether residuals are approximately normal. These diagnostics support the validity of the
regression conclusions.

In [0]:
# Select the model with the best test performance (lowest RMSE)
best_model_name = results["RMSE (test)"].idxmin()
print("Selected model based on test RMSE:", best_model_name)

# Extract the selected model
selected = {
    "Model 1 - Full": model1,
    "Model 2 - Reduced": model2,
    "Model 3 - Log Price": model3
}[best_model_name]

# Get residuals and fitted values
residuals = selected.resid
fitted = selected.fittedvalues

# Create residual diagnostic plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residual plot to check for patterns
sns.scatterplot(x=fitted, y=residuals, alpha=0.25, ax=axes[0])
axes[0].axhline(0, linestyle="--")
axes[0].set_title("Residuals vs Fitted Values")
axes[0].set_xlabel("Fitted values")
axes[0].set_ylabel("Residuals")

# Q-Q plot to check normality of residuals
sm.qqplot(residuals, line="45", fit=True, ax=axes[1])
axes[1].set_title("Q-Q Plot of Residuals")
plt.tight_layout()
plt.show()

### Insight
The selected model is **Model 1 – Full Multiple Linear Regression** because it has the lowest test RMSE. The residual and Q-Q plots should be used to assess whether any strong pattern or non-normality remains. The diagnostics suggest that the regression errors are not perfectly normal, so coefficient inference should be interpreted with some caution.


### Influential Observations

The assignment asks us to identify influential features/observations. Cook’s distance is used to identify observations that have a relatively large influence on the fitted regression. We report the largest values rather than automatically deleting them.

In [0]:
# Calculate Cook's distance to identify influential observations
influence = selected.get_influence()
cooks_d = influence.cooks_distance[0]

# Create dataframe and sort by Cook's distance
cook_df = pd.DataFrame({
    "Observation": np.arange(len(cooks_d)),
    "Cooks_Distance": cooks_d
}).sort_values("Cooks_Distance", ascending=False)

# Display top 10 most influential observations
print(cook_df.head(10))

### Insight
Cook's distance identifies a small number of highly influential observations, particularly
observations **26660** (Cook's D ≈ **87.3**) and **28578** (Cook's D ≈ **39.1**) — both far
above the conventional 4/n threshold and even above 1, meaning each single point is pulling the
fitted regression line noticeably on its own. The remaining top-10 observations have much
smaller Cook's distances (0.02-0.22). These extreme points should be investigated rather than
automatically deleted, because they may represent genuine high-value diamonds; their presence is
another reason to interpret individual regression coefficients cautiously.



In [0]:
# Extract feature importance from Model 1 based on coefficient magnitudes
coef_df = pd.DataFrame({
    "Feature": model1.params.index,
    "Coefficient": model1.params.values,
    "Abs_Coefficient": np.abs(model1.params.values),
    "P_Value": model1.pvalues.values
})

# Categorize features by their original column
def categorize_feature(feature_name):
    if feature_name == "Intercept":
        return "Intercept"
    elif "C(cut)" in feature_name:
        return "cut"
    elif "C(color)" in feature_name:
        return "color"
    elif "C(clarity)" in feature_name:
        return "clarity"
    else:
        return feature_name

coef_df["Original_Feature"] = coef_df["Feature"].apply(categorize_feature)

# Sort by absolute coefficient value
coef_df_sorted = coef_df.sort_values("Abs_Coefficient", ascending=False)

print("\n=== Most Influential Features (Model 1) - Grouped by Original Column ===")
print("\nNote: C(column)[T.value] means the dummy variable for that category level")
print("      The reference category (not shown) has coefficient = 0\n")

# Display top 15
print("\n--- TOP 15 BY ABSOLUTE COEFFICIENT ---\n")
for idx, row in coef_df_sorted.head(15).iterrows():
    sig = "***" if row["P_Value"] < 0.001 else "**" if row["P_Value"] < 0.01 else "*" if row["P_Value"] < 0.05 else ""
    print(f"{row['Original_Feature']:12s} | {row['Feature']:30s} | Coef: {row['Coefficient']:10.2f} | p={row['P_Value']:.4f} {sig}")

# Group by original feature and show summary
print("\n\n--- SUMMARY BY ORIGINAL FEATURE ---\n")
# Get all unique original features and sort them
categorical_features = ["cut", "color", "clarity"]
numerical_features = [feat for feat in coef_df["Original_Feature"].unique() 
                      if feat not in categorical_features and feat != "Intercept"]
all_features = sorted(numerical_features) + categorical_features + ["Intercept"]

for orig_feat in all_features:
    subset = coef_df[coef_df["Original_Feature"] == orig_feat].sort_values("Abs_Coefficient", ascending=False)
    if len(subset) > 0:
        print(f"\n{orig_feat.upper()}:")
        for idx, row in subset.iterrows():
            sig = "***" if row["P_Value"] < 0.001 else "**" if row["P_Value"] < 0.01 else "*" if row["P_Value"] < 0.05 else ""
            print(f"  {row['Feature']:30s}  Coef: {row['Coefficient']:10.2f}  (p={row['P_Value']:.4f}) {sig}")

print("\n*** p<0.001, ** p<0.01, * p<0.05")

## Part 3: Interpretation and Insights

### Key findings

- **EDA:** `price` is strongly right-skewed (skewness approx 1.62); most diamonds are inexpensive,
  with a long tail of expensive stones. `carat`, `x`, `y`, and `z` are also right-skewed, and
  visibly track each other and `price` closely in the scatterplots.
- **Correlation:** `carat` has by far the strongest relationship with price (Pearson r approx 0.92),
  followed by the dimension variables `x`, `y`, `z` (r approx 0.87-0.89). `table` has a weak positive
  correlation (r approx 0.13) and `depth` has essentially none (r approx -0.01) in isolation -- yet both
  turn out to be statistically significant once the other predictors are controlled for in the
  regression, showing why correlation alone should not be used to decide which variables to drop.
- **Categorical variables:** boxplots and one-way ANOVA both show that `cut`, `color`, and
  `clarity` are associated with statistically significant differences in mean price (p < 0.001
  for all three). They are retained as predictors.
- **Multicollinearity:** VIF values for `x` (1,398), `z` (664), `y` (538) and even `carat` (78)
  indicate severe multicollinearity among the size-related variables. `x`, `y`, and `z` were
  removed in Model 2 in favour of `carat` as the single representative size measure.
- **Model comparison:** on the test set, Model 1 (full) achieved R2 = 0.918, RMSE approx Rs 1,141, MAE
  approx Rs 740; Model 2 (reduced) was close behind at R2 = 0.914; Model 3 (log-price) looked strong on
  the log scale (R2 = 0.889) but collapsed to R2 = -3.146 once back-transformed to the price
  scale, due to retransformation bias amplifying a few large errors.
- **Most influential features:** `carat` dominates, followed by `clarity` and `color`; `depth`
  and `table` matter but are minor by comparison.

### Practical implications
The regression results can support pricing analysis and help  understand
which diamond characteristics command a premium: carat weight above all, then clarity and colour
grade. This can inform inventory pricing, marketing emphasis (e.g. highlighting clarity/colour
grade for smaller stones), and rough checks on whether a quoted price is in line with a stone's
characteristics.

### Limitations
- The analysis identifies statistical associations, not causal relationships.
- The dataset contains only physical/quality attributes; other real-world price drivers (brand,
  certification body, retailer margin, market/economic conditions) are not available.
- A small number of extreme, highly influential observations  were kept in the data;
  they may be genuine but could also reflect data-entry issues that cannot be verified with the
  information available.
- Ordinary least squares assumes a linear relationship and residual normality/homoscedasticity;
  the Q-Q plot and residual plot show these assumptions are not perfectly satisfied,
  so p-values and confidence intervals should be treated as approximate.

### Recommendations
Use Model 1 as an explanatory/pricing-support model rather than a fully causal or automated
pricing engine. Future work could: 

(1) collect additional variables such as brand, certification
body, and market/seller information; 

(2) apply a bias-corrected retransformation (e.g. Duan's
smearing estimator) if a log-price model is desired; 

(3) investigate the specific highly
influential observations identified by Cook's distance to check whether they are genuine or
data-entry errors; and 

(4) test non-linear or interaction terms (e.g. carat x clarity) if further
accuracy is required.



### Overall interpretation
- **Strongest numerical predictor:** `carat` because it has the strongest relationship with `price`.
- **Categorical predictors to retain:** `cut`, `color`, and `clarity`, because ANOVA shows significant price differences across their groups.
- **Variables causing multicollinearity:** `x`, `y`, and `z` (and high VIF for `carat` as well). They measure overlapping aspects of diamond size.
- **Reduced-model choice:** remove `x`, `y`, and `z` for a simpler interpretation and retain `carat`, `depth`, `table`, `cut`, `color`, and `clarity`.
- **Final predictive choice:** Model 1 is selected because it has the best test-set performance, although its individual coefficients for highly correlated variables should not be interpreted in isolation.
